In [1]:
import pandas as pd

In [9]:
df = pd.read_parquet('predictions-single_prompt/251213-predictions-single_prompt-final/final_rules_predictions.parquet')

In [10]:
df

,name,gpt5-advertising,gpt5-ai_generated,gpt5-animal_cruelty,gpt5-automated_tools_bots,gpt5-be_nice,gpt5-block_report_mute,gpt5-codes_conduct,gpt5-consequences_moderation,gpt5-copyright_piracy_credit,...,gpt5-violence_cw,gpt5-impersonation,gpt5-impersonation_duplicate_check,gpt5-copyright,gpt5-error,gpt5-error_type,gpt5-raw_response,gpt5-animal_cruelty_cw,gpt5-advertising_cw,gpt5-copyright_piracy
0,hades.town,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,None,None,None,NaN,NaN,NaN
1,porny.social,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,None,None,None,NaN,NaN,NaN
2,mesmerized.online,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,NaN,NaN,None,None,None,NaN,NaN,NaN
3,opensocial.media,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,NaN,NaN,None,None,None,NaN,NaN,NaN
4,os-tans.social,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,NaN,NaN,None,None,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8272,social.i-solutions.dev,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,None,None,None,NaN,NaN,NaN
8273,mastodon.dragonbox.de,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,None,None,None,NaN,NaN,NaN
8274,dubnation.social,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,NaN,NaN,None,None,None,NaN,NaN,NaN
8275,nog.community,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,None,None,None,NaN,NaN,NaN


## todo: map name back onto translated rules

In [11]:
df_trans = pd.read_parquet('../../final_rules_dataset_w_translations.parquet')

In [14]:
merged_df = df.merge(df_trans, on='name', how='outer')

In [16]:
merged_df.columns

Index(['name', 'gpt5-advertising', 'gpt5-ai_generated', 'gpt5-animal_cruelty',
       'gpt5-automated_tools_bots', 'gpt5-be_nice', 'gpt5-block_report_mute',
       'gpt5-codes_conduct', 'gpt5-consequences_moderation',
       'gpt5-copyright_piracy_credit', 'gpt5-crypto', 'gpt5-CSAM',
       'gpt5-cw_general', 'gpt5-dont_be_dick', 'gpt5-harassment',
       'gpt5-hate_speech', 'gpt5-illegal', 'gpt5-incitement_violence',
       'gpt5-language_specific', 'gpt5-misinfo', 'gpt5-nazis', 'gpt5-nsfw',
       'gpt5-nsfw_cw', 'gpt5-politics', 'gpt5-politics_cw',
       'gpt5-adult_content', 'gpt5-adult_content_cw', 'gpt5-privacy',
       'gpt5-private_no_reg', 'gpt5-registration_requirements',
       'gpt5-religion', 'gpt5-scrape_model_training', 'gpt5-sensitive_cw',
       'gpt5-spam', 'gpt5-spoilers', 'gpt5-spoilers_cw', 'gpt5-violence',
       'gpt5-violence_cw', 'gpt5-impersonation',
       'gpt5-impersonation_duplicate_check', 'gpt5-copyright', 'gpt5-error',
       'gpt5-error_type', 'gpt5-r

In [33]:
rows = []

for col in merged_df.columns:
    if col.startswith("gpt5-"):
        # Nur numerische Werte extrahieren
        is_numeric = pd.to_numeric(merged_df[col], errors='coerce')
        
        # Zeilen mit 1 und 0 filtern UND gültige 'translated_joined_rules'
        ones = merged_df[(is_numeric == 1) & merged_df['translated_joined_rules'].notna()]
        zeros = merged_df[(is_numeric == 0) & merged_df['translated_joined_rules'].notna()]
        
        # 5 Einträge zufällig auswählen (oder so viele wie verfügbar)
        ones_sample = ones.sample(n=min(5, len(ones)), random_state=42)
        zeros_sample = zeros.sample(n=min(5, len(zeros)), random_state=42)
        
        # Zeilen zum Ergebnis hinzufügen
        for _, row in ones_sample.iterrows():
            rows.append({
                'col': col,
                'translated_joined_rules': row['translated_joined_rules'],
                'value': 1
            })
        for _, row in zeros_sample.iterrows():
            rows.append({
                'col': col,
                'translated_joined_rules': row['translated_joined_rules'],
                'value': 0
            })

# Neuen DataFrame erstellen
sampled_df = pd.DataFrame(rows)

# Ausgabe
print(sampled_df.head(20))

                  col                            translated_joined_rules  \
0    gpt5-advertising  Respect other users: no harassment, discrimina...   
1    gpt5-advertising  no hate speech, discrimination, and/or bigotry...   
2    gpt5-advertising  conversations must be respectful. please alert...   
3    gpt5-advertising  no discrimination, including (but not limited ...   
4    gpt5-advertising  no discrimination, including (but not limited ...   
5    gpt5-advertising                                           no nazis   
6    gpt5-advertising                           be a member of my family   
7    gpt5-advertising  be a good human. if this server ever gets a cr...   
8    gpt5-advertising  Senate prohibits the content of robotic conten...   
9    gpt5-advertising  sexually explicit or violent media must be mar...   
10  gpt5-ai_generated  website operators reserve the right to decide ...   
11  gpt5-ai_generated  the responsibility for your operation exists o...   
12  gpt5-ai_